# Iterators, Generators & Decorators



## Part 1 — Iteration

## The iteration protocol

A Python *iterable* is anything that defines `__iter__`. That method returns an *iterator* — an object that defines `__next__`. Each call to `__next__` returns the next element, or raises `StopIteration` when exhausted.

```text
   for x in xs:                       <- what you write
       ...

   it = iter(xs)                      <- what Python does: get an iterator
   while True:
       try:
           x = next(it)               <- repeatedly call __next__
       except StopIteration:
           break
       ...
```

This is why `for` works on every iterable type uniformly. Lists, tuples, dicts, sets, strings, files, generators — every one defines `__iter__`. So do user classes you write.

In [ ]:
# Manual iteration — under the hood of for
xs = [10, 20, 30]
it = iter(xs)
print(it)                # <list_iterator object>

print(next(it))          # 10
print(next(it))          # 20
print(next(it))          # 30
try:
    next(it)
except StopIteration:
    print("done")

## Writing your own iterable

To make your class iterable, give it an `__iter__` method that returns an iterator. The iterator must have a `__next__` method.

You have two choices:

- **Return `self`** — make the class itself the iterator. Quick but single-pass.
- **Return a separate iterator object** — usually a generator (next section). Cleaner; supports multiple concurrent iterations of the same container.

Below: a simple `CountUp(n)` that yields `0, 1, ..., n-1`. We write it the long way to show the protocol, then immediately show the generator version that does the same thing in two lines.

In [ ]:
# Long form — explicit class with __iter__ and __next__
class CountUp:
    def __init__(self, n):
        self.n = n

    def __iter__(self):
        self.i = 0
        return self        # this object IS the iterator

    def __next__(self):
        if self.i >= self.n:
            raise StopIteration
        v = self.i
        self.i += 1
        return v

for x in CountUp(5):
    print(x, end=" ")
print()

# Manual iteration works too
print(list(CountUp(3)))

# Generator form — same behavior, three lines instead of ten
def count_up_gen(n):
    i = 0
    while i < n:
        yield i
        i += 1

print(list(count_up_gen(5)))

## Iterators are single-pass

An iterator is consumed as you iterate it. Once exhausted, it's done — you can't restart it. If you need to iterate again, get a fresh iterator from the underlying iterable.

This bites people who pass a generator to a function expecting to iterate it multiple times. The fix: materialize to a list once, or accept the iterable (not the iterator) and call `iter()` yourself.

In [ ]:
xs = [1, 2, 3]

# Lists are iterables (not iterators) — you can iterate them many times
print(list(xs))
print(list(xs))           # still works

# Iterators are iterators — single pass
it = iter(xs)
print(list(it))           # [1, 2, 3]
print(list(it))           # [] — already consumed

## `itertools` — the standard library's iterator toolbox

`itertools` ships a collection of iterator combinators — generic operations that compose with any iterable. Lazy, memory-efficient. The high-value ones:

| Function | What it does |
|---|---|
| `chain(a, b, ...)` | concatenate iterables |
| `chain.from_iterable(iters)` | flatten one level |
| `islice(iter, stop)` / `islice(iter, start, stop, step)` | slice without materializing |
| `takewhile(pred, iter)` | take from start while predicate holds |
| `dropwhile(pred, iter)` | drop from start while predicate holds |
| `filterfalse(pred, iter)` | filter, but keep ones where predicate is false |
| `accumulate(iter, op=...)` | running totals; default op is `+` |
| `groupby(iter, key=...)` | group consecutive equal-key elements |
| `product(*iters)` | Cartesian product |
| `permutations(iter, r)`, `combinations(iter, r)` | combinatorics |
| `count(start, step)` | infinite arithmetic sequence |
| `cycle(iter)` | repeat infinitely |
| `repeat(value, n)` | repeat one value n times (or forever) |
| `zip_longest(a, b, fillvalue=...)` | zip but doesn't truncate |

In [ ]:
import itertools as it

# chain — concatenate
print(list(it.chain([1, 2], [3, 4], [5])))

# islice — slice an iterator
print(list(it.islice(range(20), 5, 15, 2)))   # [5, 7, 9, 11, 13]

# takewhile / dropwhile — stop or skip based on predicate
print(list(it.takewhile(lambda n: n < 5, [1, 3, 5, 7, 2, 1])))   # [1, 3]
print(list(it.dropwhile(lambda n: n < 5, [1, 3, 5, 7, 2, 1])))   # [5, 7, 2, 1]

# accumulate — running totals
print(list(it.accumulate([1, 2, 3, 4, 5])))                       # [1, 3, 6, 10, 15]
print(list(it.accumulate([3, 1, 4, 1, 5], max)))                  # [3, 3, 4, 4, 5]

# groupby — group consecutive equal-key items (NOT sort-and-group!)
data = sorted(["apple", "ant", "bee", "ball", "cat"], key=lambda s: s[0])
for letter, group in it.groupby(data, key=lambda s: s[0]):
    print(letter, list(group))

# product, combinations — combinatorics
print(list(it.product([1, 2], ["a", "b"])))
print(list(it.combinations([1, 2, 3, 4], 2)))

## Part 2 — Generators

## `yield` — turning a function into a generator

Any function with a `yield` statement is a **generator function**. Calling it does NOT run the body — it returns a generator object. The body runs each time you advance the generator with `next()`, pausing at each `yield` and resuming on the next `next()`.

```python
def count_up(n):
    i = 0
    while i < n:
        yield i      # pause here, hand value to caller, resume on next()
        i += 1
```

The body is **paused state** — local variables persist between calls, loops and conditionals continue from exactly where they left off. This is exactly what makes generators useful for representing streams: you can write iteration logic with normal control flow, and the runtime handles the suspend/resume.

In [ ]:
def count_up(n):
    print(f"  starting, n={n}")
    i = 0
    while i < n:
        print(f"  about to yield {i}")
        yield i
        print(f"  resumed after yield {i}")
        i += 1
    print("  finished")

# Calling it does NOT run the body
g = count_up(3)
print(f"got generator: {g}")
print("---")

# Each next() advances to the next yield
print(f"first:  {next(g)}")
print(f"second: {next(g)}")
print(f"third:  {next(g)}")
try:
    next(g)
except StopIteration:
    print("done")

## Generators are memory-efficient

A generator produces values one at a time. Where a list comprehension materializes the entire result, a generator expression streams. This matters when:

- The full sequence wouldn't fit in memory.
- You only need a prefix of the sequence (the generator stops being asked for values).
- You're piping through several transformations — keeping them lazy means data flows through in chunks instead of building intermediate lists at each step.

In [ ]:
# Read a large file line by line, count those matching a condition
import io

# Simulate a multi-line "file"
data = "\n".join(f"line {i}" for i in range(1_000_000))
f = io.StringIO(data)

def lines_containing(file, needle):
    for line in file:
        if needle in line:
            yield line.rstrip()

# Memory: constant, regardless of file size
count = 0
for matching in lines_containing(f, "999"):
    count += 1
    if count > 5:
        break
print(f"matched at least {count} lines (stopped early)")

## `yield from` — delegating to another iterable

`yield from sub_iterable` yields every value from `sub_iterable`, transparently. Useful when a generator wants to forward through another iterable without an explicit loop.

```python
def chain(a, b):
    yield from a
    yield from b
```

This is the same idea as `itertools.chain`, written by hand. It also forwards exception-handling and `.send()` values, so it's the right tool when composing generators.

In [ ]:
def chain_iters(*iters):
    for it in iters:
        yield from it

print(list(chain_iters([1, 2], [3, 4], [5])))

# Flatten one level of nesting
def flatten_once(nested):
    for sublist in nested:
        yield from sublist

print(list(flatten_once([[1, 2], [3], [4, 5, 6]])))

## Modeling pipelines as chained generators

Generators compose into data pipelines that stream — no intermediate lists, constant memory. Each stage takes an iterable in, yields transformed elements out.

In [ ]:
# Pipeline: read lines -> parse ints -> drop negatives -> take first 5

def read_lines():
    for line in ["3", "-1", "5", "abc", "7", "-2", "11", "13", "17"]:
        yield line

def parse_ints(lines):
    for line in lines:
        try:
            yield int(line)
        except ValueError:
            continue

def positives(ints):
    for n in ints:
        if n >= 0:
            yield n

# Chain them — nothing runs until the final loop
pipeline = positives(parse_ints(read_lines()))

# Stop after 3 — only enough data flows through to produce the result
result = list(itertools.islice(pipeline, 3))
print(result)

import itertools

## Two-way generators — `.send()`, `.throw()`, `.close()`

Generators can receive values back from the caller via `.send(value)`. Inside the generator, the expression `yield x` evaluates to the value `send` was called with. `.throw()` injects an exception at the current yield. `.close()` raises `GeneratorExit` and shuts the generator down.

These are the building blocks of older coroutine patterns (pre-`async`/`await`). You rarely write these by hand in modern code — `asyncio` (notebook 09) is the right tool — but the mechanism exists.

In [ ]:
def echoer():
    print("  generator: started")
    while True:
        received = yield
        if received is None:
            return
        print(f"  generator: got {received!r}")

e = echoer()
next(e)               # prime the generator (advance to first yield)
e.send("hello")
e.send("world")
e.close()             # shut it down cleanly

## Part 3 — Decorators

## A decorator is a function that takes a function

The mental model: a decorator is just a higher-order function. It takes a function as input, returns a function as output. The `@decorator` syntax above a `def` is sugar for "pass this function through `decorator`, rebind the name to the result."

```python
@my_decorator
def f():
    ...

# is exactly the same as:

def f():
    ...
f = my_decorator(f)
```

That's it. Everything else is variations on this shape.

The reason to use one: wrap a function with extra behaviour — logging, timing, caching, access control, retry — without modifying the function itself, and without writing the wrapper plumbing at every call site.

In [ ]:
# A decorator that logs calls
def log_calls(func):
    def wrapper(*args, **kwargs):
        print(f"  -> {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"  <- {func.__name__} returned {result!r}")
        return result
    return wrapper

@log_calls
def add(a, b):
    return a + b

print(add(3, 4))

# The @log_calls above is exactly equivalent to:
def multiply(a, b):
    return a * b
multiply = log_calls(multiply)

print(multiply(3, 4))

## `functools.wraps` — preserve metadata

The naive wrapper above replaces `add` with `wrapper`. That means `add.__name__` becomes `"wrapper"`, `add.__doc__` is lost, and signature introspection sees the wrapper instead of the wrapped function.

`functools.wraps` is a tiny decorator-of-decorators that copies the relevant metadata from the wrapped function to the wrapper. **Always use it when writing a decorator.**

In [ ]:
from functools import wraps

def log_calls(func):
    @wraps(func)                          # preserves func's metadata
    def wrapper(*args, **kwargs):
        print(f"  -> {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_calls
def greet(name):
    """Say hello to someone."""
    return f"hello, {name}"

print(greet("ganesh"))
print(greet.__name__)        # 'greet'  — preserved
print(greet.__doc__)         # 'Say hello to someone.' — preserved

## Decorators that take arguments — the three-level shape

When you want `@retry(times=3)` instead of plain `@retry`, you need one extra level. The signature changes from "function in, function out" to "**args in, decorator out**, which itself is **function in, function out**."

```python
def retry(times):                 # outer: takes the decorator's arguments
    def decorator(func):          # middle: the actual decorator
        @wraps(func)
        def wrapper(*args, **kwargs):   # inner: the actual wrapper
            ...
        return wrapper
    return decorator
```

The shape is awkward the first time. The trick to reading it: count the levels from the outside in — each `def` is one rung, and the `@retry(times=3)` call site evaluates from the outside in too.

In [ ]:
from functools import wraps

def retry(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"  attempt {attempt + 1} failed: {e}")
            raise last_error
        return wrapper
    return decorator

import random

@retry(times=3)
def flaky():
    if random.random() < 0.7:
        raise RuntimeError("transient failure")
    return "success"

random.seed(42)
print(flaky())

## Class decorators

A class decorator is the same idea applied to a class — takes a class in, returns a class out. `@dataclass` is the standard-library example: it inspects the class's annotations and adds `__init__`, `__repr__`, `__eq__`, and friends.

You'll write your own occasionally — to register classes in a global table, to add common methods, to enforce some structure. Most of the time `@dataclass`-like decorators from libraries cover the use cases.

In [ ]:
# A class decorator that registers subclasses
handlers = {}

def register(name):
    def wrap(cls):
        handlers[name] = cls
        return cls
    return wrap

@register("click")
class ClickHandler:
    def handle(self, event):
        return f"clicked {event}"

@register("scroll")
class ScrollHandler:
    def handle(self, event):
        return f"scrolled {event}"

print(handlers)
print(handlers["click"]().handle("button"))

## Stacking decorators

You can stack multiple decorators. They apply **bottom-up**: the closest one to the function wraps first.

```python
@log_calls
@retry(times=3)
def fetch():
    ...

# is equivalent to:
def fetch():
    ...
fetch = log_calls(retry(times=3)(fetch))
```

So `retry` wraps `fetch` first, then `log_calls` wraps the result of `retry`. At runtime, every call goes through `log_calls` first, then through the `retry` wrapper, then into `fetch`. The print-then-retry semantics are usually what you want.

## Common standard-library decorators

Worth recognizing:

| Decorator | Module | What it does |
|---|---|---|
| `@property` | builtin | turn a method into an attribute |
| `@staticmethod` | builtin | method that doesn't take `self`/`cls` |
| `@classmethod` | builtin | method that takes `cls` instead of `self` |
| `@dataclass` | dataclasses | generate `__init__`, `__repr__`, `__eq__` |
| `@cache` | functools | memoize — unbounded cache |
| `@lru_cache(maxsize=...)` | functools | memoize with size limit |
| `@cached_property` | functools | compute once, cache as instance attribute |
| `@singledispatch` | functools | type-based dispatch |
| `@wraps(func)` | functools | preserve metadata when writing decorators |
| `@contextmanager` | contextlib | generator → context manager (notebook 06) |
| `@fixture` | pytest | declare a test fixture (notebook 10) |

In [ ]:
from functools import cache, lru_cache, cached_property, singledispatch
import time

# @cache — unbounded memoization
@cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(50))            # near-instant thanks to memoization

# @cached_property — computed once per instance
class Profile:
    def __init__(self, name):
        self.name = name

    @cached_property
    def expensive(self):
        print(f"  computing for {self.name}...")
        time.sleep(0.1)
        return f"result for {self.name}"

p = Profile("ganesh")
print(p.expensive)         # computes
print(p.expensive)         # cached — no recompute

# @singledispatch — type-based dispatch
@singledispatch
def render(value):
    return f"default: {value}"

@render.register
def _(value: int):
    return f"int: {value}"

@render.register
def _(value: list):
    return f"list of {len(value)}"

print(render("hello"))
print(render(42))
print(render([1, 2, 3]))

## When NOT to decorate

The readability cliff:

- **One call site.** If a function is only called from one place, just write the wrapping logic inline. Decorators pay off when the same wrapping fires in many places.
- **Behaviour that fundamentally changes the function's contract.** A decorator that turns `def f(x): ...` into something that takes different arguments will confuse every reader.
- **Hidden state.** A decorator that introduces global state (caches, registries) makes test isolation harder. Be explicit about it.
- **Order matters and is non-obvious.** Stacked decorators interact in ways that aren't always intuitive. Keep stacks short.

Decorators are a force multiplier when they capture a real cross-cutting concern — logging, timing, retry, caching. They are a footgun when used to clever-hide ordinary logic.